# Proyecto C: Curva de luz de un púlsar

Un púlsar es una estrella que emite un pulso de luz que se repite con un periodo muy regular. En los datos entregados hay 1000 periodos, pero el ruido es tan grande que el pulso no se ve a simple vista.

La idea del proyecto es **plegar** los datos (*folding*): poner todos los periodos uno encima del otro y calcular el promedio. Después, al perfil promedio se le ajusta una curva.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# El archivo csv debe estar en la misma carpeta que el notebook
datos = pd.read_csv("datos_C_curvaluz_pulsar.csv")
t = datos["Time_sec"].to_numpy()      # tiempo [s]
flujo = datos["Flux"].to_numpy()      # flujo medido

print("Cantidad de datos:", flujo.size)
print("Datos faltantes:", np.isnan(flujo).sum())
print("Paso de tiempo:", t[1] - t[0], "s")

## Folding (plegado)

El paso de tiempo es 0,001 s y cada periodo tiene 1000 puntos, por lo tanto el periodo del púlsar es

$$P = 1000 \cdot 0{,}001\ \text{s} = 1\ \text{s}.$$

Los datos se ordenan en una matriz de 1000 filas (los periodos) y 1000 columnas (la fase dentro del periodo). Al promediar por columnas se obtiene el perfil del pulso.

**¿Por qué sirve promediar?** El pulso es igual en todos los periodos, así que al promediar se mantiene. El ruido en cambio es aleatorio: unas veces suma y otras resta. Al promediar $N$ periodos el ruido baja aproximadamente $\sqrt{N}$ veces. Con $N=1000$ el ruido baja unas $\sqrt{1000}\approx 32$ veces.

In [ ]:
n_periodos = 1000
n_puntos = 1000

matriz = flujo.reshape(n_periodos, n_puntos)   # cada fila es un periodo
fase = np.arange(n_puntos) / n_puntos          # fase entre 0 y 1

perfil = matriz.mean(axis=0)                   # promedio de todos los periodos

# Se mide el ruido en una zona sin pulso (fase menor a 0,3)
ruido_un_periodo = matriz[0, fase < 0.3].std()
ruido_promedio = perfil[fase < 0.3].std()

print("Ruido en un solo periodo:", round(ruido_un_periodo, 3))
print("Ruido en el perfil promedio:", round(ruido_promedio, 4))
print("Cuántas veces bajó el ruido:", round(ruido_un_periodo / ruido_promedio, 1))
print("Valor esperado (raíz de 1000):", round(np.sqrt(n_periodos), 1))

## Ajuste del perfil

El perfil tiene un solo pulso con forma de campana, por eso se ajusta una función gaussiana más una constante:

$$S(\phi) = A\, e^{-\frac{(\phi-\mu)^2}{2\sigma^2}} + c$$

- $A$: altura del pulso
- $\mu$: posición del pulso (fase del centro)
- $\sigma$: ancho del pulso
- $c$: nivel de fondo

El ajuste se hace con `curve_fit` de SciPy, que busca los valores de los parámetros que dejan la curva lo más cerca posible de los datos (mínimos cuadrados).

In [ ]:
def gaussiana(x, A, mu, sigma, c):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + c

# Valores iniciales: se estiman mirando el gráfico del perfil
inicial = [0.7, 0.5, 0.03, 0.0]

parametros, covarianza = curve_fit(gaussiana, fase, perfil, p0=inicial)
errores = np.sqrt(np.diag(covarianza))      # incertidumbre de cada parámetro

nombres = ["A", "mu", "sigma", "c"]
for nombre, valor, error in zip(nombres, parametros, errores):
    print(f"{nombre} = {valor:.4f} +/- {error:.4f}")

## Gráficos

Se muestran dos gráficos: una parte de los datos en bruto (3 periodos) y el perfil promedio con la curva ajustada.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.0), constrained_layout=True)

# Datos en bruto: primeros 3 segundos (3 periodos)
mascara = t < 3
ax[0].plot(t[mascara], flujo[mascara], color="gray", linewidth=0.6)
ax[0].set_xlabel("Tiempo [s]")
ax[0].set_ylabel("Flujo")
ax[0].set_title("Datos en bruto (3 periodos)")

# Perfil promedio y ajuste
x = np.linspace(0, 1, 500)
ax[1].plot(fase, perfil, ".", color="gray", markersize=3, label="Perfil promedio")
ax[1].plot(x, gaussiana(x, *parametros), color="red", label="Ajuste gaussiano")
ax[1].set_xlabel("Fase")
ax[1].set_ylabel("Flujo promedio")
ax[1].set_title("Perfil plegado y ajuste")
ax[1].legend()

fig.savefig("figuras/pulsar_resultados.pdf")
fig.savefig("figuras/pulsar_resultados.png", dpi=200)
plt.show()

## Varios pulsos con el ruido superpuesto

Para ver mejor cuánto tapa el ruido a la señal, se grafican 5 periodos seguidos. En gris están los datos medidos (señal más ruido) y en rojo el pulso ajustado, repetido en cada periodo. Así se ve el tamaño del pulso comparado con el ruido.

In [ ]:
mascara5 = t < 5                      # primeros 5 segundos = 5 periodos
fase5 = (t[mascara5] / 1.0) % 1.0     # fase de cada punto (periodo P = 1 s)
pulsos = gaussiana(fase5, *parametros)  # pulso ajustado repetido en cada periodo

fig2, ax2 = plt.subplots(figsize=(10, 2.5), constrained_layout=True)
ax2.plot(t[mascara5], flujo[mascara5], color="gray", linewidth=0.6, label="Datos medidos (pulso + ruido)")
ax2.plot(t[mascara5], pulsos, color="red", linewidth=1.8, label="Pulso ajustado")
ax2.set_xlabel("Tiempo [s]")
ax2.set_ylabel("Flujo")
ax2.set_title("Cinco periodos: pulso ajustado sobre el ruido")
ax2.legend(loc="upper right")

fig2.savefig("figuras/pulsar_pulsos_ruido.pdf")
fig2.savefig("figuras/pulsar_pulsos_ruido.png", dpi=200)
plt.show()